# OOD Evaluation: CHAT Corpus (Single-Utterance)

This notebook runs lightweight OOD annotation on any CHAT corpus (`.cha`) using a single-utterance setup.

What it does:
- prepares utterance-level JSONL from a corpus directory
- runs `utterance_only` inference only (no context-mode comparison)
- builds a flat review CSV

Set `CORPUS_DIR` in the config cell to your target corpus path.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shamira-venturini/talkbank-morphosyntax-error-annotator.git"
REPO_NAME = "talkbank-morphosyntax-error-annotator"
COLAB_REPO_ROOT = Path("/content") / REPO_NAME

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "scripts").exists() and (cur / "data").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Could not locate repository root from current working directory")

if Path("/content").exists():
    if not COLAB_REPO_ROOT.exists():
        clone_cmd = ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_ROOT)]
        print("Cloning repository:", " ".join(clone_cmd))
        subprocess.run(clone_cmd, check=True)
    repo_root = COLAB_REPO_ROOT
else:
    repo_root = find_repo_root(Path.cwd())

os.chdir(repo_root)
print("Repo root:", repo_root)
print("Python:", sys.executable)

In [ ]:
# Install runtime dependencies in Colab.
!pip -q install -U transformers peft bitsandbytes accelerate sentencepiece

In [ ]:
# Load HF token from Colab secrets when available.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set. Add it to Colab Secrets or os.environ."

In [ ]:
# Config
CORPUS_DIR = "data/OOD_data/Vercellotti"  # change to any CHAT corpus folder
CORPUS_NAME = ""  # optional override; empty => folder name
OUTPUT_PREFIX = ""  # optional override; empty => slug from corpus name
PREPARED_DIR = "data/processed/ood_chat"
RESULTS_DIR = "results/ood_chat"

SPEAKER_POLICY = "dominant"      # dominant | first_participant | all
BATCH_SIZE = 8
MAX_NEW_TOKENS = 48
MAX_SEQ_LENGTH = 320
MAX_CONTEXT_CHARS = 1200
LIMIT = 0

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
ADAPTER_REPO = "mash-mash/talkbank-morphosyntax-annotator-final-recon_full_comp_preserve_final_seed3407"
CHAT_TOKENS = "experiments/recon_full_comp_preserve/chat_tokens.json"
STAGE3_SPLIT = "experiments/recon_full_comp_preserve/stage3_train.jsonl"


In [ ]:
# 1) Prepare utterance-level OOD dataset from .cha files
import json

prep_cmd = [
    "python3", "scripts/prepare_ood_vercellotti_dataset.py",
    "--corpus-dir", CORPUS_DIR,
    "--out-dir", PREPARED_DIR,
    "--speaker-policy", SPEAKER_POLICY,
    "--min-word-count", "1",
]
if CORPUS_NAME:
    prep_cmd.extend(["--corpus-name", CORPUS_NAME])
if OUTPUT_PREFIX:
    prep_cmd.extend(["--output-prefix", OUTPUT_PREFIX])
print(" ".join(prep_cmd))
subprocess.run(prep_cmd, check=True)

prep_summary = json.loads((Path(PREPARED_DIR) / "summary.json").read_text(encoding="utf-8"))
prepared_jsonl = Path(prep_summary["outputs"]["utterances_jsonl"])
assert prepared_jsonl.exists(), f"Missing prepared file: {prepared_jsonl}"
print("Prepared utterances:", prepared_jsonl)
print("Rows:", prep_summary.get("utterances_kept"))


In [ ]:
# 2) Run OOD inference (utterance_only only, lightweight)
cmd = [
    "python3", "scripts/run_ood_context_inference.py",
    "--input-jsonl", str(prepared_jsonl),
    "--out-dir", RESULTS_DIR,
    "--context-mode", "utterance_only",
    "--batch-size", str(BATCH_SIZE),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--max-context-chars", str(MAX_CONTEXT_CHARS),
    "--base-model", BASE_MODEL,
    "--adapter-repo", ADAPTER_REPO,
    "--chat-tokens", CHAT_TOKENS,
    "--stage3-split", STAGE3_SPLIT,
]
if LIMIT > 0:
    cmd.extend(["--limit", str(LIMIT)])
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 3) Build review CSV from single-utterance predictions
predictions_jsonl = Path(RESULTS_DIR) / "predictions_utterance_only.jsonl"
review_csv = Path(RESULTS_DIR) / "review_utterance_only.csv"

review_cmd = [
    "python3", "scripts/build_ood_review_csv.py",
    "--predictions-jsonl", str(predictions_jsonl),
    "--out-csv", str(review_csv),
]
print(" ".join(review_cmd))
subprocess.run(review_cmd, check=True)
print("Review CSV:", review_csv)


In [ ]:
# 4) Quick preview of outputs
import pandas as pd

review_df = pd.read_csv(review_csv)
print("Rows:", len(review_df))
print("prediction_status counts:")
print(review_df["prediction_status"].value_counts(dropna=False))
print("\nlabels_or_clean top:")
print(review_df["labels_or_clean"].value_counts(dropna=False).head(20))
display(review_df.head(10))


In [ ]:
# Optional: zip results for local download
import shutil
from google.colab import files

zip_path = shutil.make_archive(str(Path(RESULTS_DIR)), "zip", root_dir=str(Path(RESULTS_DIR).parent), base_dir=Path(RESULTS_DIR).name)
print("Created:", zip_path)
files.download(zip_path)
